In [1]:
import torch
from pathlib import Path

## loading data

In [3]:
tokens_dir = Path("../model-comparison/my_results/tokens")
token_pt_files = sorted(tokens_dir.glob("*.pt"))

tokens = {}
for f in token_pt_files:
    tokens[f.stem] = torch.load(f)

In [15]:
import torch
import numpy as np
import pandas as pd
from scipy.stats import levene, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

BASE_DIR = Path("../model-comparison/my_results/tokens")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")

    #df.to_csv(BASE_DIR / f"variance_per_layer_{orig_name}__vs__{shuf_name}.csv", index=False)



=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000160
Mann–Whitney p (orig < shuf): 6.872e-12
Layers with FDR<0.05: 27/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000153
Mann–Whitney p (orig < shuf): 3.256e-12
Layers with FDR<0.05: 29/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000145
Mann–Whitney p (orig < shuf): 3.250e-12
Layers with FDR<0.05: 32/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000146
Mann–Whitney p (orig < shuf): 2.959e-11
Layers with FDR<0.05: 27/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000155
Mann–Whitney p (or

In [16]:
BASE_DIR = Path("../model-comparison/my_results/words")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")


=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000153
Mann–Whitney p (orig < shuf): 1.800e-11
Layers with FDR<0.05: 28/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000151
Mann–Whitney p (orig < shuf): 3.252e-12
Layers with FDR<0.05: 29/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000118
Mann–Whitney p (orig < shuf): 3.252e-12
Layers with FDR<0.05: 29/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000168
Mann–Whitney p (orig < shuf): 1.883e-11
Layers with FDR<0.05: 29/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000152
Mann–Whitney p (or

In [17]:
BASE_DIR = Path("../model-comparison/my_results/sentences")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")


=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000067
Mann–Whitney p (orig < shuf): 7.830e-02
Layers with FDR<0.05: 6/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000056
Mann–Whitney p (orig < shuf): 2.030e-03
Layers with FDR<0.05: 2/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000036
Mann–Whitney p (orig < shuf): 5.654e-02
Layers with FDR<0.05: 19/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000068
Mann–Whitney p (orig < shuf): 3.560e-01
Layers with FDR<0.05: 0/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000062
Mann–Whitney p (orig 